## Imports 

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

## Find project directory

In [3]:
PROJECT_ROOT = Path.cwd()

# Find the project root automatically
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / "README.md").exists() and (parent / "data").exists():
        PROJECT_ROOT = parent
        break

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Data Warehouse Multiclass.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = PROCESSED_DIR / "cleaned_dataset.csv"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw dataset:")
print(RAW_PATH)

print("\nProcessed directory:")
print(PROCESSED_DIR)

Project root:
/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler

Raw dataset:
/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/raw/Data Warehouse Multiclass.csv

Processed directory:
/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed


## Verify and load dataset

In [4]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{RAW_PATH}\n\n"
        "Make sure the CSV file is inside data/raw/"
    )

print("Dataset found successfully.")

Dataset found successfully.


In [5]:
df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.
Rows    : 280,985
Columns : 39


In [6]:
original_rows = len(df)
original_columns = df.shape[1]

print(f"Original rows    : {original_rows:,}")
print(f"Original columns : {original_columns}")

Original rows    : 280,985
Original columns : 39


## Standardize column names

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Standardized column names:")
print(df.columns.tolist())

Standardized column names:
['composite_key', 'age_level', 'gender', 'bmi_level', 'smoking', 'diabetes', 'age', 'age_normalized', 'bmi', 'hypertension', 'heart_disease', 'hba1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status', 'source_dataset', 'disease_flags', 'sublabel', 'label']


In [8]:
object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    df[col] = df[col].str.strip()

print(f"Processed {len(object_columns)} text columns.")

/tmp/ipykernel_13622/1661356490.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns


Processed 19 text columns.


In [9]:
df = df.replace(r"^\s*$", np.nan, regex=True)

print("Empty-string values converted to NaN.")

Empty-string values converted to NaN.


## Missing value analysis

In [10]:
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
)

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary)

No missing values found.


## Duplicate analysis

In [11]:
duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.4f}%"
)

Exact duplicate rows: 2,284
Duplicate percentage: 0.8129%


In [12]:
df_before_duplicates = len(df)

df = df.drop_duplicates().reset_index(drop=True)

df_after_duplicates = len(df)

print(f"Rows before duplicate removal : {df_before_duplicates:,}")
print(f"Rows after duplicate removal  : {df_after_duplicates:,}")
print(f"Rows removed                  : {df_before_duplicates - df_after_duplicates:,}")

Rows before duplicate removal : 280,985
Rows after duplicate removal  : 278,701
Rows removed                  : 2,284


In [13]:
remaining_duplicates = df.duplicated().sum()

print(
    f"Remaining exact duplicate rows: "
    f"{remaining_duplicates:,}"
)

Remaining exact duplicate rows: 0


## Candidate target columns

In [14]:
candidate_target_columns = [
    "diabetes",
    "hypertension",
    "heart_disease",
    "bmi_level",
    "disease_flags",
    "sublabel",
    "label"
]

available_targets = [
    col for col in candidate_target_columns
    if col in df.columns
]

print("Candidate target/label-related columns:")
for col in available_targets:
    print(f" - {col}")

Candidate target/label-related columns:
 - diabetes
 - hypertension
 - heart_disease
 - bmi_level
 - disease_flags
 - sublabel
 - label


## Inspect every candidate target

In [15]:
for col in available_targets:

   
    print(f"COLUMN: {col}")
    print("_" * 80 + "\n")

    print("Data type:", df[col].dtype)
    print("Unique values:", df[col].nunique(dropna=False))

    counts = (
        df[col]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    counts["percentage"] = (
        counts["count"] / len(df) * 100
    )

    display(counts.head(50))

COLUMN: diabetes
________________________________________________________________________________

Data type: str
Unique values: 2


,count,percentage
diabetes,,
No,178083,63.89751
Yes,100618,36.10249


COLUMN: hypertension
________________________________________________________________________________

Data type: float64
Unique values: 137


,count,percentage
hypertension,,
0.000000,104044,37.331764
1.000000,8060,2.891988
0.333333,3597,1.290630
0.142857,3215,1.153566
0.217391,3141,1.127014
0.166667,2519,0.903836
0.200000,2319,0.832075
0.111111,2296,0.823822
0.057722,2061,0.739502


COLUMN: heart_disease
________________________________________________________________________________

Data type: float64
Unique values: 125


...

,count,percentage
disease_flags,,
"0,0,0",104648,37.548484
"0,0,1",69787,25.040097
"1,0,1",66472,23.850650
"1,0,0",31903,11.447035
"0,1,0",2583,0.926800
"1,1,0",1398,0.501613
"0,1,1",1065,0.382130
"1,1,1",845,0.303192


COLUMN: sublabel
________________________________________________________________________________

Data type: str
Unique values: 8


,count,percentage
sublabel,,
N,104648,37.548484
HY,69787,25.040097
DI_HY,66472,23.850650
DI,31903,11.447035
HT,2583,0.926800
DI_HT,1398,0.501613
HT_HY,1065,0.382130
DI_HT_HY,845,0.303192


COLUMN: label
________________________________________________________________________________

Data type: str
Unique values: 2


,count,percentage
label,,
Abnormal,174053,62.451516
Normal,104648,37.548484


## Diabetes analysis

In [16]:
if "diabetes" in df.columns:

    print("DIABETES DISTRIBUTION")

    diabetes_dist = (
        df["diabetes"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    diabetes_dist["percentage"] = (
        diabetes_dist["count"] / len(df) * 100
    )

    display(diabetes_dist)

DIABETES DISTRIBUTION


,count,percentage
diabetes,,
No,178083,63.89751
Yes,100618,36.10249


## Hypertension analysis

In [17]:
if "hypertension" in df.columns:

    print("HYPERTENSION DISTRIBUTION")

    hypertension_dist = (
        df["hypertension"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    hypertension_dist["percentage"] = (
        hypertension_dist["count"] / len(df) * 100
    )

    display(hypertension_dist)

HYPERTENSION DISTRIBUTION


,count,percentage
hypertension,,
0.000000,104044,37.331764
1.000000,8060,2.891988
0.333333,3597,1.290630
0.142857,3215,1.153566
0.217391,3141,1.127014
...,...,...
0.211864,240,0.086114
0.003086,230,0.082526
0.000300,199,0.071403


## Heart disease analysis

In [18]:
if "heart_disease" in df.columns:

    print("HEART DISEASE DISTRIBUTION")

    heart_dist = (
        df["heart_disease"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    heart_dist["percentage"] = (
        heart_dist["count"] / len(df) * 100
    )

    display(heart_dist)

HEART DISEASE DISTRIBUTION


,count,percentage
heart_disease,,
0.000000,119385,42.836230
1.000000,3910,1.402937
0.222222,3845,1.379615
0.032258,3074,1.102974
0.080000,3040,1.090775
...,...,...
0.000776,255,0.091496
0.218220,240,0.086114
0.000300,199,0.071403


## Obesity/BMI level analysis

In [19]:
if "bmi_level" in df.columns:

    print("BMI LEVEL DISTRIBUTION")

    bmi_level_dist = (
        df["bmi_level"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    bmi_level_dist["percentage"] = (
        bmi_level_dist["count"] / len(df) * 100
    )

    display(bmi_level_dist)

BMI LEVEL DISTRIBUTION


,count,percentage
bmi_level,,
Obese,98377,35.298402
Overweight,76897,27.591218
Normal,70591,25.328578
Underweight,32836,11.781802


## Suspicious columns

In [20]:
if "disease_flags" in df.columns:

    print("DISEASE FLAGS")

    display(
        df["disease_flags"]
        .value_counts(dropna=False)
        .head(50)
        .to_frame("count")
    )

DISEASE FLAGS


,count
disease_flags,
"0,0,0",104648
"0,0,1",69787
"1,0,1",66472
"1,0,0",31903
"0,1,0",2583
"1,1,0",1398
"0,1,1",1065
"1,1,1",845


In [21]:
if "sublabel" in df.columns:

    print("SUBLABEL")

    display(
        df["sublabel"]
        .value_counts(dropna=False)
        .head(50)
        .to_frame("count")
    )

SUBLABEL


,count
sublabel,
N,104648
HY,69787
DI_HY,66472
DI,31903
HT,2583
DI_HT,1398
HT_HY,1065
DI_HT_HY,845


In [22]:
if "label" in df.columns:

    print("LABEL")

    display(
        df["label"]
        .value_counts(dropna=False)
        .head(50)
        .to_frame("count")
    )

LABEL


,count
label,
Abnormal,174053
Normal,104648


In [23]:
if "source_dataset" in df.columns:

    print("SOURCE DATASET")

    source_dist = (
        df["source_dataset"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    source_dist["percentage"] = (
        source_dist["count"] / len(df) * 100
    )

    display(source_dist)

SOURCE DATASET


,count,percentage
source_dataset,,
hypertension,174982,62.784848
diabetes,93844,33.671928
heart,9875,3.543224


## Duplicate analysis by composite key

In [24]:
if "composite_key" in df.columns:

    unique_keys = df["composite_key"].nunique(dropna=False)

    print(f"Total rows: {len(df):,}")
    print(f"Unique composite keys: {unique_keys:,}")
    print(
        f"Average rows per unique key: "
        f"{len(df) / unique_keys:.2f}"
    )

Total rows: 278,701
Unique composite keys: 192
Average rows per unique key: 1451.57


In [25]:
if "composite_key" in df.columns:

    key_counts = (
        df["composite_key"]
        .value_counts()
        .to_frame("row_count")
    )

    print("Composite keys appearing more than once:")

    display(
        key_counts[key_counts["row_count"] > 1]
        .head(30)
    )

Composite keys appearing more than once:


,row_count
composite_key,
Middle_Female_Overweight_Never_No,5600
Mature_Female_Overweight_Never_No,5415
Middle_Female_Obese_Never_No,5020
Mature_Female_Obese_Never_No,4986
Middle_Female_Normal_Never_No,4806
Mature_Male_Overweight_Never_No,4418
Middle_Male_Overweight_Never_No,4385
Young_Female_Normal_Never_No,3833
Mature_Male_Obese_Never_No,3716


## BMI leakage investigation

In [26]:
if {"bmi", "bmi_level"}.issubset(df.columns):

    print("BMI statistics by BMI level")

    display(
        df.groupby("bmi_level")["bmi"]
        .agg(
            ["count", "min", "mean", "median", "max", "std"]
        )
        .sort_values("mean")
    )

BMI statistics by BMI level


,count,min,mean,median,max,std
bmi_level,,,,,,
Underweight,32836,10.01,16.628742,16.70,18.499041,1.151952
Normal,70591,18.50,21.831664,21.90,24.998857,1.863172
Overweight,76897,25.00,27.405728,27.32,29.998623,1.244845
Obese,98377,30.00,35.247881,34.90,95.690000,3.857603


In [27]:
if {"bmi", "bmi_level"}.issubset(df.columns):

    bmi_bins = pd.qcut(
        df["bmi"],
        q=10,
        duplicates="drop"
    )

    bmi_crosstab = pd.crosstab(
        df["bmi_level"],
        bmi_bins
    )

    display(bmi_crosstab)

bmi,"(10.009, 17.95]","(17.95, 20.75]","(20.75, 23.26]","(23.26, 25.66]","(25.66, 27.32]","(27.32, 28.8]","(28.8, 31.31]","(31.31, 34.2]","(34.2, 37.2]","(37.2, 95.69]"
bmi_level,,,,,,,,,,
Normal,0,22954,27848,19789,0,0,0,0,0,0
Obese,0,0,0,0,0,0,14777,28612,27196,27792
Overweight,0,0,0,8080,39078,17236,12503,0,0,0
Underweight,27887,4949,0,0,0,0,0,0,0,0


## Feature exclusion investigation

In [28]:
metadata_or_suspicious_columns = [
    "disease_flags",
    "sublabel",
    "label",
    "source_dataset",
    "composite_key"
]

available_exclusions = [
    col for col in metadata_or_suspicious_columns
    if col in df.columns
]

print("Initially excluded from ML features:")
for col in available_exclusions:
    print(f" - {col}")

Initially excluded from ML features:
 - disease_flags
 - sublabel
 - label
 - source_dataset
 - composite_key


In [29]:
feature_candidates = [
    col
    for col in df.columns
    if col not in available_exclusions
]

print(f"Number of preliminary feature candidates: {len(feature_candidates)}")

print("\nFeatures:")
for col in feature_candidates:
    print(f" - {col}")

Number of preliminary feature candidates: 34

Features:
 - age_level
 - gender
 - bmi_level
 - smoking
 - diabetes
 - age
 - age_normalized
 - bmi
 - hypertension
 - heart_disease
 - hba1c_level
 - glucose
 - cholesterol
 - sleep_hours
 - triglycerides
 - physical_activity
 - family_history
 - stress_level
 - low_hdl_cholesterol
 - high_ldl_cholesterol
 - blood_pressure
 - high_blood_pressure
 - sugar_consumption
 - crp_level
 - homocysteine_level
 - systolic_bp
 - diastolic_bp
 - alcohol_intake
 - salt_intake
 - heart_rate
 - hdl
 - ldl
 - education_level
 - employment_status


In [30]:
missing_cells = df.isna().sum().sum()

print(f"Total missing cells: {missing_cells:,}")

Total missing cells: 0


In [31]:
df.to_csv(CLEAN_PATH, index=False)

print("Cleaned dataset saved successfully:")
print(CLEAN_PATH)

print("\nShape:", df.shape)

Cleaned dataset saved successfully:
/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/cleaned_dataset.csv

Shape: (278701, 39)


In [32]:

print("FINAL CLEANING REPORT")
print("_" * 80+"\n")

print(f"Original rows:              {original_rows:,}")
print(f"Rows after duplicate removal:{len(df):,}")
print(f"Rows removed:               {original_rows - len(df):,}")
print(f"Columns:                    {df.shape[1]:,}")
print(f"Remaining exact duplicates: {df.duplicated().sum():,}")
print(f"Missing cells:              {df.isna().sum().sum():,}")

print("\nCandidate targets:")
for col in available_targets:
    print(f" - {col}")

print("\nSuspicious/metadata columns:")
for col in available_exclusions:
    print(f" - {col}")

print("\nSaved file:")
print(CLEAN_PATH)

FINAL CLEANING REPORT
________________________________________________________________________________

Original rows:              280,985
Rows after duplicate removal:278,701
Rows removed:               2,284
Columns:                    39
Remaining exact duplicates: 0
Missing cells:              0

Candidate targets:
 - diabetes
 - hypertension
 - heart_disease
 - bmi_level
 - disease_flags
 - sublabel
 - label

Suspicious/metadata columns:
 - disease_flags
 - sublabel
 - label
 - source_dataset
 - composite_key

Saved file:
/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/cleaned_dataset.csv
